In [ ]:
import pandas as pd
import numpy as np

print("Environment is ready")
print("Pandas:", pd.__version__)
print("Numpy:", np.__version__)

In [ ]:
from pathlib import Path

data_path = Path("../data/raw/dcgm.csv")

dcgm = pd.read_csv(data_path)

print("shape:", dcgm.shape)
print("\nColumns:")
print(dcgm.columns.tolist())

## Check data Quality

In [ ]:
print("Missing values:")
print(dcgm.isna().sum())

print("\nDuplicate rows:")
print(dcgm.duplicated().sum())

In [ ]:
dcgm.head()

In [ ]:
dcgm.describe().T

## load the scheduler data

In [ ]:
scheduler = pd.read_csv("../data/raw/scheduler_data.csv")

print("Shape:", scheduler.shape)
print("\nColumns:")
print(scheduler.columns.tolist())


In [ ]:
scheduler["state"].value_counts()

In [ ]:
scheduler["job_type"].value_counts(dropna=False)

In [ ]:
scheduler[["cpus_req", "mem_req", "gres_req", "nodes_alloc", "priority"]].head(10)

In [ ]:
scheduler.head().T

In [ ]:
scheduler["mem_req"].describe()

In [ ]:
special_mem = scheduler["mem_req"] > 9e18

print("Special mem_req values:", special_mem.sum())
print("Percentage:", special_mem.mean() * 100)

In [ ]:
print("GPU requested:", scheduler["gres_req"].notna().sum())
print("No GPU request:", scheduler["gres_req"].isna().sum())

print("\nGPU request types:")
print(scheduler["gres_req"].value_counts(dropna=False).head(15))

#Comparing two data sets with respect to "id_job"

In [ ]:
common_data = set(dcgm["id_job"]) & set(scheduler["id_job"])

print("DCGM JOBS: ", dcgm["id_job"].nunique())
print("DCGM JOBS: ", dcgm["id_job"].unique())
print("SCHEDULER JOBS: ", scheduler["id_job"].nunique())
print("SCHEDULER JOBS: ", scheduler["id_job"].unique())

print("TOTAL COMMON LENGTH ", len(common_data))

In [ ]:
dcgm_coverage = len(common_data)/ dcgm["id_job"].nunique() *100
scheduler_coverage = len(common_data)/ scheduler["id_job"].nunique() *100

print("Total DCGM COVERAGE IS : ", dcgm_coverage)
print("Total SCHEDULER_COVERAGE IS ", scheduler_coverage)

In [ ]:
print("Total number of duplicated job ids in dcgm dataset: ", dcgm["id_job"].duplicated().sum())
print("Total number of duplicated job ids in scheduler dataset: ", scheduler["id_job"].duplicated().sum())

In [ ]:
print(dcgm["id_job"].value_counts().head(25))

In [ ]:
job_id = 49748333192

print(dcgm[dcgm["id_job"] == job_id][["id_job", "Node", "gpu_id"]])

In [ ]:
job_data = dcgm[dcgm["id_job"] == job_id]
print("DCGM records: ", len(job_data))
print("Unique nodes: ", job_data["Node"].nunique())
print("Unique GPU IDS: ", job_data["gpu_id"].nunique())
print("NODE- GPU combinations : ", job_data[["Node", "gpu_id"]].drop_duplicates().shape[0])

In [ ]:
scheduler[scheduler["id_job"] == job_id][["id_job", "gres_req", "gres_alloc", "gres_used", "nodes_alloc", "cpus_req"]]

In [ ]:
gpu_count_per_job = (
    dcgm.groupby("id_job")
        .apply(lambda x: x[["Node", "gpu_id"]].drop_duplicates().shape[0])
)

print(gpu_count_per_job.value_counts().sort_index())

In [ ]:
##Count actual GPUs from DCGM
gpu_count_dcgm =(
    dcgm.groupby("id_job").apply(lambda x: x[["Node", "gpu_id"]].drop_duplicates().shape[0]).rename("gpu_count_dcgm")
)

# Take scheduler data for the same jobs
scheduler_gpu = scheduler[scheduler["id_job"].isin(gpu_count_dcgm.index)][["id_job", "gres_alloc"]].drop_duplicates("id_job")

#Join Both 
gpu_validation = scheduler_gpu.join(gpu_count_dcgm, on="id_job")

print(gpu_validation.head(20))

In [ ]:
gpu_validation["gres_gpu_count"] = (
    gpu_validation["gres_alloc"].str.extract("r(\d+)", expand = False).astype(float)
)

gpu_validation["match"] = (
    gpu_validation["gres_gpu_count"] == gpu_validation["gpu_count_dcgm"]
)

print(gpu_validation["match"].value_counts())

In [ ]:
print(gpu_validation[
    ["gres_alloc", "gres_gpu_count", "gpu_count_dcgm"]
].head(10))

print("\nData types:")
print(gpu_validation[
    ["gres_alloc", "gres_gpu_count", "gpu_count_dcgm"]
].dtypes)

print("\nMissing values:")
print(gpu_validation[
    ["gres_gpu_count", "gpu_count_dcgm"]
].isna().sum())

In [ ]:
gpu_validation["gres_gpu_count"] = (
    gpu_validation["gres_alloc"]
    .str.split(":")
    .str[-1]
    .astype(int)
)

gpu_validation["match"] = (
    gpu_validation["gres_gpu_count"]
    == gpu_validation["gpu_count_dcgm"]
)

print(gpu_validation["match"].value_counts())

In [ ]:
mismatches = gpu_validation[
    ~gpu_validation["match"]
]

print(
    mismatches[
        ["id_job", "gres_alloc", "gres_gpu_count", "gpu_count_dcgm"]
    ]
)

In [ ]:
job_id = mismatches.iloc[0]["id_job"]

print("Job ID:", job_id)

print("\nScheduler:")
print(
    scheduler[scheduler["id_job"] == job_id][
        ["id_job", "gres_req", "gres_alloc", "nodes_alloc",
         "cpus_req", "time_submit", "time_start", "time_end"]
    ]
)

print("\nDCGM:")
print(
    dcgm[dcgm["id_job"] == job_id][
        ["id_job", "Node", "gpu_id", "powerusage_watts_avg",
         "energyconsumed_joules", "totalexecutiontime_sec"]
    ]
)

In [ ]:
job_dcgm = dcgm[dcgm["id_job"] == job_id]

print("DCGM rows:", len(job_dcgm))
print("Unique nodes:", job_dcgm["Node"].nunique())
print("Unique Node-GPU combinations:",
      job_dcgm[["Node", "gpu_id"]].drop_duplicates().shape[0])

print("\nGPU IDs:")
print(job_dcgm["gpu_id"].value_counts())

print("\nNodes:")
print(job_dcgm["Node"].nunique())

In [ ]:
job_id = dcgm["id_job"].value_counts().index[0]

job_data = dcgm[dcgm["id_job"] == job_id]

print("Job ID:", job_id)
print("GPU records:", len(job_data))
print("\nSelected columns:")
print(
    job_data[
        [
            "energyconsumed_joules",
            "powerusage_watts_avg",
            "powerusage_watts_max",
            "avgsmutilization_pct",
            "avgmemoryutilization_pct",
            "totalexecutiontime_sec"
        ]
    ].head()
)